In [1]:
!pip install gurobipy

In [2]:
import numpy as np
import cvxpy as cp
import time


def build_and_solve_misocp(I, J, f_j, d_ij, big_lambda_i,
                            kappa_j, v_j, M_j, p_j,
                            epsilon=2.0, rho_bar=0.999,
                            tighten_bounds=True,
                            add_valid_inequalities=True,
                            time_limit=None, mip_gap=None,
                            verbose=True):
    """
    Algorithm 2: Monolithic MISOCP benchmark for (P-MISOCP).

    Capacity cost is quadratic: c_j(mu_j) = kappa_j * mu_j^2.

    Parameters
    ----------
    I, J : int
        Number of demand zones and candidate stations.
    f_j : array (J,)      Fixed opening cost.
    d_ij : array (I,J)    Travel cost per unit demand.
    big_lambda_i : array (I,) Exogenous zone demand.
    kappa_j : array (J,)  Quadratic capacity-cost coefficient.
    v_j : array (J,)      Queueing congestion weight.
    M_j : array (J,)      Max service capacity when open.
    p_j : array (J,)      Linear revenue-in-lambda_hat coefficient.
    epsilon : float       Stability margin.
    rho_bar : float       Max utilization (rho < 1 strictly; paper's rho-bar).

    Returns :
    dict with solution, bounds, runtime, gap, node count.
    """

    total_demand = float(np.sum(big_lambda_i))

    # Decision variables
    x_j = cp.Variable(J, boolean=True)
    y_ij = cp.Variable((I, J), nonneg=True)
    lam_bar_j = cp.Variable(J, nonneg=True)        # routed demand (lambda-bar)
    lam_hat_j = cp.Variable(J, nonneg=True)        # sqrt(admitted demand)
    mu_j = cp.Variable(J, nonneg=True)             # service rate
    rho_j = cp.Variable(J, nonneg=True)            # utilization
    q_j = cp.Variable(J, nonneg=True)              # queueing epigraph
    tc_j = cp.Variable(J, nonneg=True)             # capacity-cost epigraph
    tr_j = cp.Variable(J)                          # negative-revenue epigraph (can be negative)

    constraints = []

    # Routing constraints
    constraints += [cp.sum(y_ij, axis=1) <= 1]
    constraints += [y_ij <= cp.reshape(x_j, (1, J), order='C')]
    constraints += [lam_bar_j == y_ij.T @ big_lambda_i]
    constraints += [y_ij <= 1]

    # Step 2: Tighten variable bounds
    if tighten_bounds:
        # lambda_bar_j can never exceed total system demand
        constraints += [lam_bar_j <= total_demand]
        # lambda_hat_j^2 <= lambda_bar_j  ->  lambda_hat_j <= sqrt(total_demand)
        constraints += [lam_hat_j <= np.sqrt(total_demand)]
        # mu_j bounded by M_j even before tying to x_j (helps presolve)
        constraints += [mu_j <= M_j]
        # rho_j is a utilization ratio, strictly less than 1 in steady state
        constraints += [rho_j <= rho_bar]

    # Step 3: Valid inequalities
    if add_valid_inequalities:
        # Can't route more demand to station j than its capacity supports
        constraints += [lam_bar_j <= cp.multiply(M_j, x_j)]

    # lambda_hat_j^2 <= lambda_bar_j
    constraints += [cp.square(lam_hat_j) <= lam_bar_j]

    # lambda_hat_j^2 + eps*x_j <= mu_j
    constraints += [cp.square(lam_hat_j) + epsilon * x_j <= mu_j]
    constraints += [mu_j <= cp.multiply(M_j, x_j)]

    # ---- Rotated SOC constraints (Proposition 3.6) -------------------------
    # (rho_j, mu_j, sqrt(2) lambda_hat_j) in Q^3_r  <=>  rho_j*mu_j >= lambda_hat_j^2
    # DCP-compliant form: quad_over_lin(lambda_hat_j, mu_j) <= rho_j  (mu_j > 0)
    constraints += [cp.quad_over_lin(lam_hat_j[k], mu_j[k]) <= rho_j[k] for k in range(J)]

    # (q_j, 1-rho_j, sqrt(2) rho_j) in Q^3_r  <=>  q_j*(1-rho_j) >= rho_j^2
    # i.e. q_j >= rho_j^2 / (1-rho_j) = quad_over_lin(rho_j, 1-rho_j)
    constraints += [cp.quad_over_lin(rho_j[k], 1 - rho_j[k]) <= q_j[k] for k in range(J)]

    # tc_j >= kappa_j * mu_j^2
    constraints += [kappa_j[k] * cp.square(mu_j[k]) <= tc_j[k] for k in range(J)]

    # tr_j >= -p_j * lambda_hat_j
    # r_hat_j(lambda_hat_j) = p_j * lambda_hat_j (linear => concave => SOC-representable)
    constraints += [tr_j[k] >= -p_j[k] * lam_hat_j[k] for k in range(J)]

    # Objective
    travel_cost = d_ij * big_lambda_i[:, np.newaxis]
    objective = cp.Minimize(
        cp.sum(cp.multiply(f_j, x_j))
        + cp.sum(cp.multiply(travel_cost, y_ij))
        + cp.sum(cp.multiply(v_j, q_j))
        + cp.sum(tc_j)
        + cp.sum(tr_j)
    )

    prob = cp.Problem(objective, constraints)

    solver_opts = {}
    if time_limit is not None:
        solver_opts['TimeLimit'] = time_limit
    if mip_gap is not None:
        solver_opts['MIPGap'] = mip_gap

    t0 = time.time()
    result = prob.solve(solver=cp.GUROBI, verbose=verbose, **solver_opts)
    runtime = time.time() - t0

    # Step 5: Record runtime, node count, bounds, incumbent, gap
    gurobi_model = prob.solver_stats.extra_stats if prob.solver_stats else None
    node_count, mip_gap, best_bound, gurobi_runtime = None, None, None, None
    if gurobi_model is not None:
        try:
            node_count = gurobi_model.NodeCount
            mip_gap = gurobi_model.MIPGap
            best_bound = gurobi_model.ObjBound
            gurobi_runtime = gurobi_model.Runtime
        except Exception:
            pass

    output = {
        "status": prob.status,
        "objective_value": result,
        "runtime_sec": runtime,
        "gurobi_runtime_sec": gurobi_runtime,
        "node_count": node_count,
        "best_bound": best_bound,
        "mip_gap": mip_gap,
        "x_j": x_j.value,
        "y_ij": y_ij.value,
        "lambda_bar_j": lam_bar_j.value,
        "lambda_hat_j": lam_hat_j.value,
        "mu_j": mu_j.value,
        "rho_j": rho_j.value,
        "q_j": q_j.value,
        "tc_j": tc_j.value,
        "tr_j": tr_j.value,
        "solver_stats": prob.solver_stats,
    }
    return output

In [3]:
import numpy as np
import sys
sys.path.insert(0, '.')

# Toy instance analogous to the notebook's example
I = 1
J = 2
f_j = np.array([8.0, 10.0])
d_ij = np.array([[10.0], [1.5]])
big_lambda_i = np.array([15.0])
kappa_j = np.array([0.4, 0.5])
v_j = np.array([0.8, 0.7])
M_j = np.array([40.0, 15.0])
p_j = np.array([50.0, 50.0])   # linear-in-lambda_hat revenue coefficient (per station)
epsilon = 2.0

out = build_and_solve_misocp(
    I, J, f_j, d_ij, big_lambda_i,
    kappa_j, v_j, M_j, p_j,
    epsilon=epsilon, verbose=False
)

print("status:", out["status"])
print("objective_value:", out["objective_value"])
print("runtime_sec:", out["runtime_sec"])
print("gurobi_runtime_sec:", out["gurobi_runtime_sec"])
print("node_count:", out["node_count"])
print("best_bound:", out["best_bound"])
print("mip_gap:", out["mip_gap"])
print("x_j:", out["x_j"])
print("y_ij:\n", out["y_ij"])
print("lambda_bar_j:", out["lambda_bar_j"])
print("lambda_hat_j:", out["lambda_hat_j"])
print("mu_j:", out["mu_j"])
print("rho_j:", out["rho_j"])
print("q_j:", out["q_j"])
print("tc_j:", out["tc_j"])
print("tr_j:", out["tr_j"])

Restricted license - for non-production use only - expires 2027-11-29
status: optimal
objective_value: -63.39240126965743
runtime_sec: 0.07885885238647461
gurobi_runtime_sec: 0.012810945510864258
node_count: 1.0
best_bound: -63.39585991000041
mip_gap: 5.45592259282492e-05
x_j: [1. 1.]
y_ij:
 [[0.17330651 0.15939666]]
lambda_bar_j: [2.59959764 2.39094992]
lambda_hat_j: [1.61232677 1.54626967]
mu_j: [4.59959771 4.39094993]
rho_j: [0.5651796  0.54451776]
q_j: [0.734622  0.6509792]
tc_j: [8.46251989 9.64022103]
tr_j: [-80.61633864 -77.31348355]


In [4]:
import numpy as np
import sys
sys.path.insert(0, '.')

I = 2
J = 2
f_j = np.array([8.0, 8.0])
d_ij = np.array([[1.0, 5.0],
                  [5.0, 1.0]])
big_lambda_i = np.array([10.0, 10.0])
kappa_j = np.array([0.3, 0.3])
v_j = np.array([0.8, 0.8])
M_j = np.array([20.0, 20.0])
p_j = np.array([40.0, 40.0])
epsilon = 2.0

out_with_vi = build_and_solve_misocp(
    I, J, f_j, d_ij, big_lambda_i, kappa_j, v_j, M_j, p_j,
    epsilon=epsilon, add_valid_inequalities=True, verbose=False
)
out_without_vi = build_and_solve_misocp(
    I, J, f_j, d_ij, big_lambda_i, kappa_j, v_j, M_j, p_j,
    epsilon=epsilon, add_valid_inequalities=False, verbose=False
)

for label, out in [("WITH valid inequalities", out_with_vi),
                    ("WITHOUT valid inequalities", out_without_vi)]:
    print(f"--- {label} ---")
    print("status:", out["status"])
    print("objective_value:", out["objective_value"])
    print("node_count:", out["node_count"], " mip_gap:", out["mip_gap"])
    print("x_j:", out["x_j"])
    print("y_ij:\n", out["y_ij"])
    print()

--- WITH valid inequalities ---
status: optimal
objective_value: -129.24710970294848
node_count: 1.0  mip_gap: 5.613335547348795e-05
x_j: [1. 1.]
y_ij:
 [[7.70263091e-01 6.10876893e-11]
 [6.10876893e-11 7.70263067e-01]]

--- WITHOUT valid inequalities ---
status: optimal
objective_value: -129.24710970294848
node_count: 1.0  mip_gap: 5.613335547348795e-05
x_j: [1. 1.]
y_ij:
 [[7.70263091e-01 6.10876893e-11]
 [6.10876893e-11 7.70263067e-01]]

